# 🎬 Audio-Automation: Chatterbox V3 GPU Studio & Auto YouTube Publisher
### Channel: Garib YT (@garib_yt_official)
- **Genre:** 💔 Pure High-Stakes Desi Drama (Family sacrifice, betrayal, love & emotional twist; zero horror)
- **TTS Engine:** Chatterbox V3 Multilingual (GPU) with Step 11 Cinema Quality Radio DSP & Dubstudio CUDA flow clamp
- **Video Engine:** 1080p Full HD Video with Smartphone Music Player Center Scrolling Title (no background patti, pure white bold text, black shadow) + Glowing 16:9 Climax Thumbnail + Bottom Dialogue Subtitles + Cyan Audio Waveform + Progress Bar
- **Direct YouTube Upload:** Automated high-speed chunked upload to channel `Garib YT` via YouTube Data API v3


In [ ]:
# 1. Environment & GPU Setup
import os, sys, torch

assert torch.cuda.is_available(), '❌ GPU must be enabled in Kaggle Settings (Tesla T4 or P100)!'
print(f'🚀 Kaggle GPU Detected: {torch.cuda.get_device_name(0)}')

# Optimization for PyTorch CUDA memory allocation (Dubstudio standard)
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TTS_ENGINE'] = 'chatterbox'
os.environ['CHANNEL_NAME'] = 'Garib YT'

# API Secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ['GEMINI_API_KEY'] = user_secrets.get_secret('GEMINI_API_KEY')
except Exception:
    if not os.environ.get('GEMINI_API_KEY'):
        os.environ['GEMINI_API_KEY'] = 'AQ.Ab8RN6yQW47w5o4ZJqR1tM3sK7vP9bC2xL8eY0uN6dF1'

print('✅ Environment configured for Garib YT Drama Studio.')


In [ ]:
# 2. Install System & Audio/ML Dependencies (Python 3.13 Compatible Stack)
import subprocess, sys, os

print('📦 Installing system libraries & Hindi fonts...')
!sudo apt-get update -qq || true
!sudo apt-get install -y -qq ffmpeg sox libsox-dev build-essential python3-dev fonts-lohit-deva fonts-dejavu-core fonts-noto > /dev/null

print('📦 Upgrading build tools...')
!pip install -q --upgrade pip wheel setuptools

print('📦 Installing Chatterbox, Transformers & Audio Packages...')
!pip install -q --no-cache-dir conformer==0.3.2 pykakasi==2.3.0 pyloudnorm s3tokenizer diffusers==0.29.0 'safetensors>=0.8.0'
!pip install -q --no-cache-dir resemble-perth sox vocos einops spacy-pkuseg soundfile pydub requests beautifulsoup4 google-generativeai python-dotenv 'scipy>=1.13.0' 'librosa>=0.10.0' edge-tts google-api-python-client google-auth-oauthlib google-auth-httplib2 Pillow
!pip install -q --no-cache-dir --no-deps git+https://github.com/resemble-ai/chatterbox.git

print('📦 Ensuring compatible safetensors & transformers...')
!pip install -q --no-cache-dir 'safetensors>=0.8.0' 'transformers>=4.48.0'

# Python 3.12/3.13 Kaggle compatibility patch (Dubstudio fix)
import pkgutil, zipimport, importlib.machinery
if not hasattr(pkgutil, 'ImpImporter'):
    pkgutil.ImpImporter = zipimport.zipimporter
if hasattr(importlib.machinery, 'FileFinder'):
    importlib.machinery.FileFinder.find_module = lambda self, fullname, *args, **kwargs: self.find_spec(fullname).loader if self.find_spec(fullname) is not None else None

print('\n🔍 Verifying library imports...')
import numpy as np, scipy, torch
print(f'  torch: {torch.__version__}, cuda: {torch.cuda.is_available()}')
print(f'  numpy: {np.__version__}')
print(f'  scipy: {scipy.__version__}')

from chatterbox.mtl_tts import ChatterboxMultilingualTTS
print('✅ Chatterbox Multilingual V3 successfully imported without errors!')


In [ ]:
# 3. Clone & Initialize Audio-Automation Studio
import os, sys, json, base64, shutil
from pathlib import Path

repo_dir = Path('/kaggle/working/Audio-automation')
if not repo_dir.exists():
    print('📥 Cloning latest Audio-automation repository from GitHub...')
    !git clone https://github.com/9329238475x-wq/Audio-automation.git /kaggle/working/Audio-automation
else:
    print('🔄 Updating existing repository...')
    !cd /kaggle/working/Audio-automation && git pull

os.chdir(str(repo_dir))
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

# Configure Active YouTube Channel: Garib YT (@garib_yt_official)
tokens_dir = repo_dir / 'tokens'
tokens_dir.mkdir(parents=True, exist_ok=True)

token_b64 = 'eyJwcm9maWxlIjogImdhcmliX3l0IiwgImNoYW5uZWxfaWQiOiAiVUNiN3VFV0ZReUM2SGNQaUtaR2lrTktnIiwgImNoYW5uZWxfdGl0bGUiOiAiR2FyaWIgWVQgIiwgImN1c3RvbV91cmwiOiAiQGdhcmliX3l0X29mZmljaWFsIiwgInRodW1ibmFpbCI6ICJodHRwczovL3l0My5nZ3BodC5jb20vQWY5ZXItMnlqcjhOUXBUWDdzNGgzWXNBNmlyNTFWa0YwaDhxWndEQWxmelJhVkRfcEVkMWxEakhnMGdoZzlsay1sRm5iSU9TenFzPXM4MDAtYy1rLWMweDAwZmZmZmZmLW5vLXJqIiwgInN1YnNjcmliZXJfY291bnQiOiAiOTUiLCAidmlkZW9fY291bnQiOiAiNiIsICJ0b2tlbiI6ICJ5YTI5LmEwQVgwN0NtdGRoM2VpN2hYZ0tyUXpkaUdSc04xbl8zYnNVTjQ0TThTXzV5LVNXUXJoSExaUnV1VkhJVjBDUFVaSDczeXVUeGZNamFpWExraE1CT29UVnNNRXlhYlZlSUVURHZlSTEzMkI2blRpVkVralBEcm02LWtpV1NYNXZ4YjB2TlkwQ0dQTXcyM2Y1YkNqemRrWjV3QnQyOW4wN1U5OTBYMzFGRnZvek9NRW1jRTNOdHZZdU5PbndrU0JQWmtMTFRRYWdBZ2FDZ1lLQVdBU0FSVVNGUUhHWDJNaXJzRERtcWVJVEwzY2g5SXVsWFZ4R2cwMjA2IiwgInJlZnJlc2hfdG9rZW4iOiAiMS8vMGc0d3pWeHdBcmwya0NnWUlBUkFBR0JBU053Ri1MOUlyWFBqZTk3ZmRueUZWSmJCV0o5aHFQdGE2dGhUMTZ2eVU4RVpKQS1nVjNCWnBlS056bkFoX0VIeFJmNTItS3ZoRkF3ZyIsICJ0b2tlbl91cmkiOiAiaHR0cHM6Ly9vYXV0aDIuZ29vZ2xlYXBpcy5jb20vdG9rZW4iLCAiY2xpZW50X2lkIjogIjE4ODQ4NDI5ODA4Ni11azd1aGR1dW04b2k5N2puZG9uaTIxaHRta3RrZWdvaC5hcHBzLmdvb2dsZXVzZXJjb250ZW50LmNvbSIsICJjbGllbnRfc2VjcmV0IjogIkdPQ1NQWC1KdzNBS0xhV0ZtWWdEeE9QUU5xdTNUb2pKYnduIiwgInNjb3BlcyI6IFsiaHR0cHM6Ly93d3cuZ29vZ2xlYXBpcy5jb20vYXV0aC95b3V0dWJlLnVwbG9hZCIsICJodHRwczovL3d3dy5nb29nbGVhcGlzLmNvbS9hdXRoL3lvdXR1YmUucmVhZG9ubHkiXX0='
garib_token_data = json.loads(base64.b64decode(token_b64.encode('ascii')).decode('utf-8'))

token_file = tokens_dir / 'token_garib_yt.json'
with open(token_file, 'w', encoding='utf-8') as f:
    json.dump(garib_token_data, f, indent=2)

active_channel_file = repo_dir / 'config' / 'active_channel.txt'
active_channel_file.parent.mkdir(parents=True, exist_ok=True)
active_channel_file.write_text('garib_yt', encoding='utf-8')

print('✅ Studio configured for Channel: Garib YT (@garib_yt_official)')
print(f'✅ Character Voice Samples: {len(list((repo_dir / "voice_samples").glob("*.wav")))} studio reference voices loaded!')


In [ ]:
# 4. Generate Drama Story, Synthesize with Chatterbox V3 & Render 1080p Video
from pipeline import run_audio_story_pipeline
from story_engine.auto_drama_hunter import AutoDramaHunter

# Topic: आखिरी ख़त और वो बारिश की रात (High-stakes heartbreak and emotional twist drama)
# Hero: कबीर | Heroine: आरुषि
print('🎙️ Launching Chatterbox V3 Neural Audio & 1080p Video Pipeline on GPU...')
master_wav, master_mp3, master_mp4 = run_audio_story_pipeline(
    topic='आखिरी ख़त और वो बारिश की रात',
    hero_name='कबीर',
    heroine_name='आरुषि',
    duration_mins=5,
    output_prefix='akhiri_khat_garib_yt',
    engine='chatterbox',
    render_video=True,
    channel_name='Garib YT'
)

print('\n' + '=' * 65)
print('🎉 MASTER 1080P VIDEO RENDERED SUCCESSFULLY!')
print(f'  Master MP4: {master_mp4}')
print('=' * 65)


In [ ]:
# 5. Direct Upload to YouTube Channel: Garib YT (@garib_yt_official)
from pathlib import Path
from uploader.youtube_uploader import YouTubeUploader
from seo_engine.viral_metadata_generator import ViralMetadataGenerator

print('=' * 65)
print('🚀 LAUNCHING AUTOMATIC YOUTUBE UPLOAD TO GARIB YT')
print('=' * 65)

video_path = Path(master_mp4)
assert video_path.exists(), f'Video file not found: {video_path}'

title = '💔 आखिरी ख़त और वो बारिश की रात | Heart Touching Hindi Love Story | Garib YT'
desc = 'सुनिए कबीर और आरुषि की एक ऐसी दर्दभरी और रूह कंपा देने वाली प्रेम कहानी जो आपके दिल को छू जाएगी। जब तीन साल बाद एक अनसुलझा ख़त सामने आता है, तो प्यार और क़ुर्बानी की ऐसी सच्चाई खुलती है जिसे सुनकर आपकी आँखें भर आएँगी।\n\n🎧 कृपया हेडफ़ोन (Headphones) का उपयोग करें।\n📺 Channel: Garib YT (@garib_yt_official)'
tags = ['hindi love story', 'hindi kahaniya', 'pocket fm story', 'kuku fm stories', 'audio story hindi', 'garib yt', 'heart touching love story']

try:
    meta_gen = ViralMetadataGenerator()
    if hasattr(meta_gen, 'generate_metadata'):
        meta = meta_gen.generate_metadata(story_title='आखिरी ख़त और वो बारिश की रात', hero='कबीर', heroine='आरुषि', channel_name='Garib YT')
        title = meta.get('title', title)
        desc = meta.get('description', desc)
        tags = meta.get('tags', tags)
    elif hasattr(meta_gen, 'generate_youtube_kit'):
        kit = meta_gen.generate_youtube_kit('आखिरी ख़त और वो बारिश की रात')
        title = kit.get('viral_titles', [title])[0]
        desc = kit.get('viral_description', desc)
        tags = kit.get('viral_tags', tags)
except Exception as e:
    print(f'Metadata generation notice: {e}. Using viral defaults.')

# Locate thumbnail
thumb_path = video_path.parent / f'{video_path.stem}.jpg'
if not thumb_path.exists():
    possible_thumbs = list(video_path.parent.glob('*.jpg')) + list(Path('output/masters').glob('*.jpg'))
    thumb_path = possible_thumbs[0] if possible_thumbs else None

print(f'📺 Channel: Garib YT (@garib_yt_official)')
print(f'🎬 Video: {video_path.name} ({video_path.stat().st_size / (1024*1024):.2f} MB)')
print(f'📌 Title: {title}')
print(f'🖼️ Thumbnail: {thumb_path}')

uploader = YouTubeUploader(channel_profile='garib_yt')

def upload_progress(chunk_num, bytes_sent, total_bytes, message):
    if total_bytes > 0:
        pct = (bytes_sent / total_bytes) * 100
        print(f'  [Upload] {pct:.1f}% ({bytes_sent / (1024*1024):.1f}/{total_bytes / (1024*1024):.1f} MB) - {message}')
    else:
        print(f'  [Upload] {message}')

result = uploader.upload_video(
    video_path=video_path,
    title=title,
    description=desc,
    tags=tags,
    privacy_status='public',
    category_id='24',
    thumbnail_path=thumb_path,
    progress_callback=upload_progress
)

if result.get('success'):
    yt_url = result.get('youtube_url', f"https://youtu.be/{result.get('video_id')}")
    print('\n' + '=' * 65)
    print('🎉 VIDEO IS LIVE ON YOUTUBE!')
    print(f'🔗 YouTube Link: {yt_url}')
    print(f'📺 Channel: Garib YT (@garib_yt_official)')
    print('=' * 65)
else:
    print(f'❌ Upload result: {result}')
